Purpose of this file is to save down relevant information from the W&B runs that constitute the data for the paper. These will become Supplementary Tables.

In [45]:
# Import Required Libraries
import logging
import os

import pandas as pd

import wandb

# Setup Logging and Configuration
logging.basicConfig(
    format="%(asctime)s %(levelname)-8s [%(name)s] %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)
logger = logging.getLogger(__name__)

In [46]:
# Define WandB project and sweep details
PROJECT_NAME = "millergw/prostate_met_status"

# # Define directories for saving results
RESULTS_DIR = "../results"
joint_simu_result_savepath = os.path.join(RESULTS_DIR, "joint_simulation_prediction_metrics_per_run.csv")
single_gene_simu_result_savepath = os.path.join(RESULTS_DIR, "single_gene_spike_in_simulation_prediction_metrics_per_run.csv")
p1000_result_savepath = os.path.join(RESULTS_DIR, "p1000_empirical_prediction_metrics_per_run.csv")
# Define paths for grouped summary results
grouped_joint_simu_result_savepath = os.path.join(RESULTS_DIR, "joint_simulation_prediction_metrics_per_group.csv")
grouped_single_gene_simu_result_savepath = os.path.join(RESULTS_DIR, "single_gene_spike_in_simulation_prediction_metrics_per_group.csv")
grouped_p1000_result_savepath = os.path.join(RESULTS_DIR, "p1000_empirical_prediction_metrics_per_group.csv")

# make directory if it doesn't exist
os.makedirs(os.path.join(RESULTS_DIR), exist_ok=True)

In [47]:
# add tags to final runs
# Initialize W&B API
api = wandb.Api() 
columns_to_retrieve = [
    "run_id", "run_name", "model_type", 
    "datasets",
    "sample_binary", "n_samples",  "sigma", "odds_ratio", "num_class1_samples", "num_class0_samples", "control_frequency",
    "deltaMuGenes", "mod0_genes", "mod1_genes",
    "save_dir",
]

performance_metric_columns_to_retrieve = [
    "train_average_precision_score", "validation_average_precision_score", "test_average_precision_score",
    "train_roc_auc_score", "validation_roc_auc_score", "test_roc_auc_score",
    "train_f1_score", "validation_f1_score", "test_f1_score", 
    "train_balanced_acc", "validation_balanced_acc", "test_balanced_acc", 
    "train_acc", "validation_acc", "test_acc",
    "train_confusion_matrix", "validation_confusion_matrix", "test_confusion_matrix",
]

columns_to_retrieve.extend(performance_metric_columns_to_retrieve)

logging.info(f"Retrieving data from {len(columns_to_retrieve)} columns: {columns_to_retrieve}")

performance_metric_col_order = [
    "train_avg_precision", "validation_avg_precision", "test_avg_precision",
    "train_roc_auc_score", "validation_roc_auc_score", "test_roc_auc_score",
    "train_f1_score", "validation_f1_score", "test_f1_score", 
    "train_balanced_acc", "validation_balanced_acc", "test_balanced_acc", 
    "train_acc", "validation_acc", "test_acc",
    # "train_confusion_matrix", "validation_confusion_matrix", "test_confusion_matrix",
]

2025-09-03 19:00:47 INFO     [root] Retrieving data from 33 columns: ['run_id', 'run_name', 'model_type', 'datasets', 'sample_binary', 'n_samples', 'sigma', 'odds_ratio', 'num_class1_samples', 'num_class0_samples', 'control_frequency', 'deltaMuGenes', 'mod0_genes', 'mod1_genes', 'save_dir', 'train_average_precision_score', 'validation_average_precision_score', 'test_average_precision_score', 'train_roc_auc_score', 'validation_roc_auc_score', 'test_roc_auc_score', 'train_f1_score', 'validation_f1_score', 'test_f1_score', 'train_balanced_acc', 'validation_balanced_acc', 'test_balanced_acc', 'train_acc', 'validation_acc', 'test_acc', 'train_confusion_matrix', 'validation_confusion_matrix', 'test_confusion_matrix']


In [ ]:
def collapse_over_replicates(df, group_by_cols):
    # Average over seeds
    # Add a counts column for each group
    df_counts = df.groupby(group_by_cols).size().reset_index(name="count")
    df_avg = df.groupby(group_by_cols, as_index=False).mean()
    df_avg = df_avg.merge(df_counts, on=group_by_cols)
    return df_avg

def make_grouped_summary_with_mean_and_stdev(df, param_cols, metric_cols):
    # 1) Count per group
    counts = df.groupby(param_cols).size().reset_index(name="count")

    # 2) Mean & std per group
    summary = (
        df.groupby(param_cols)[metric_cols]
        .agg(['mean', 'std'])
    )

    # Flatten multi-index columns
    summary.columns = [f"{m}_{stat}" for m, stat in summary.columns]
    summary = summary.reset_index()

    # 3) Merge counts back in
    summary_numeric = pd.merge(counts, summary, on=param_cols)

    # 4) Formatted as "mean ± std"
    summary_formatted = summary_numeric.assign(**{
        m: summary_numeric[f"{m}_mean"].round(3).astype(str) 
        + " ± " + summary_numeric[f"{m}_std"].round(3).astype(str)
        for m in metric_cols
    })[param_cols + ["count"] + metric_cols]
    return summary_formatted

# Joint sampling

In [49]:
all_records = []
runs = api.runs(f"{PROJECT_NAME}", filters={"tags": "2D-simu-20250822"})
    # "$and": [{"tags": "pnet-simu-paper-20250822"}, {"tags": "1D-simu-20250822"}, {"tags": "2D-simu-20250822"}, {"tags": "p1000-20250822"}]})
logging.info(f"Working on joint simulation results: {len(runs)} runs found")

for run in runs: 
    config = run.config
    summary = run.summary
    all_records.append({
        "run_id": run.id,
        "run_name": run.name,
        "model_type": config.get("model_type"),
        "sample_binary": config.get("sample_binary"),
        "n_samples": config.get("num_samples"),
        "datasets": config.get("datasets"),
        "sigma": float(config.get("sigma")),
        "odds_ratio": float(config.get("odds_ratio")),
        "save_dir": config.get("save_dir"),
        "num_class1_samples": config.get("num_class1_samples"),
        "num_class0_samples": config.get("num_class0_samples"),
        "deltaMuGenes": config.get("deltaMuGenes"),
        "mod0_genes": config.get("mod0_genes"),
        "mod1_genes": config.get("mod1_genes"),
    })
    for split in ["train", "validation", "test"]:
        all_records[-1][f"{split}_roc_auc_score"] = summary.get(f"{split}_roc_auc_score")
        all_records[-1][f"{split}_balanced_acc"] = summary.get(f"{split}_balanced_acc")
        all_records[-1][f"{split}_avg_precision"] = summary.get(f"{split}_average_precision_score")
        all_records[-1][f"{split}_acc"] = summary.get(f"{split}_acc")
        all_records[-1][f"{split}_f1_score"] = summary.get(f"{split}_f1_score")
        # all_records[-1][f"{split}_confusion_matrix"] = summary.get(f"{split}_confusion_matrix")
    for split in ["validation", "test"]:
        all_records[-1][f"{split}_feature_importances_path"] = os.path.join(config.get("save_dir"), f"{split}_gene_feature_importances.csv")
        all_records[-1][f"{split}_gene_importances_path"] = os.path.join(config.get("save_dir"), f"{split}_gene_importances.csv")

logging.debug("Convert to a single DataFrame")
df = pd.DataFrame(all_records)

group_by_cols = ["model_type", "odds_ratio", "sigma", "sample_binary", "n_samples"]
logging.debug(f'Add a unique group identifer column by joining together the unique identifiers: {group_by_cols}')
df["group_identifier"] = df.apply(
    lambda row: f"OR-{row['odds_ratio']}_sigma-{row['sigma']}_nSamples-{row['n_samples']}_sampleBinary-{row['sample_binary']}",
    axis=1)

logging.debug("Changing the column order for the final DataFrame")
col_order = group_by_cols + performance_metric_col_order
df = df[col_order]

logging.debug("Make results DF collapsed over replicates")
df_group_summary = make_grouped_summary_with_mean_and_stdev(df, param_cols=group_by_cols, metric_cols=performance_metric_col_order)

logging.info(f"Saving the joint simulation results DataFrame to CSV at {joint_simu_result_savepath}")
df.to_csv(joint_simu_result_savepath, float_format="%.3f", index=False)
df_group_summary.to_csv(grouped_joint_simu_result_savepath, index=False)

display(df.round(3).head(2))
display(df_group_summary.head(2))


2025-09-03 19:00:48 INFO     [root] Working on joint simulation results: 640 runs found
2025-09-03 19:00:51 INFO     [root] Saving the joint simulation results DataFrame to CSV at ../results/joint_simulation_prediction_metrics_per_run.csv


,model_type,odds_ratio,sigma,sample_binary,n_samples,train_avg_precision,validation_avg_precision,test_avg_precision,train_roc_auc_score,validation_roc_auc_score,test_roc_auc_score,train_f1_score,validation_f1_score,test_f1_score,train_balanced_acc,validation_balanced_acc,test_balanced_acc,train_acc,validation_acc,test_acc
0,pnet,1.0,0.0,True,10000,0.551,0.483,0.497,0.559,0.487,0.502,0.571,0.507,0.549,0.546,0.489,0.517,0.546,0.489,0.517
1,pnet,1.0,0.1,True,10000,0.563,0.517,0.506,0.579,0.524,0.513,0.585,0.565,0.535,0.557,0.531,0.509,0.557,0.531,0.509


,model_type,odds_ratio,sigma,sample_binary,n_samples,count,train_avg_precision,validation_avg_precision,test_avg_precision,train_roc_auc_score,...,test_roc_auc_score,train_f1_score,validation_f1_score,test_f1_score,train_balanced_acc,validation_balanced_acc,test_balanced_acc,train_acc,validation_acc,test_acc
0,pnet,1.0,0.0,False,1000,10,0.513 ± 0.062,0.514 ± 0.035,0.515 ± 0.045,0.502 ± 0.076,...,0.5 ± 0.041,0.413 ± 0.278,0.426 ± 0.273,0.421 ± 0.281,0.498 ± 0.047,0.504 ± 0.015,0.503 ± 0.022,0.498 ± 0.047,0.504 ± 0.015,0.503 ± 0.022
1,pnet,1.0,0.0,False,10000,10,0.572 ± 0.007,0.505 ± 0.01,0.496 ± 0.011,0.583 ± 0.005,...,0.494 ± 0.017,0.561 ± 0.046,0.506 ± 0.06,0.494 ± 0.057,0.558 ± 0.004,0.502 ± 0.014,0.494 ± 0.014,0.558 ± 0.004,0.502 ± 0.014,0.494 ± 0.014


# Single-gene spike-in simulations

In [50]:
all_records = []
runs = api.runs(f"{PROJECT_NAME}", filters={"tags": "1D-simu-20250822"})
    # "$and": [{"tags": "pnet-simu-paper-20250822"}, {"tags": "1D-simu-20250822"}, {"tags": "2D-simu-20250822"}, {"tags": "p1000-20250822"}]})
logging.info(f"Working on single-gene simulation results: {len(runs)} runs found")

for run in runs: 
    config = run.config
    summary = run.summary
    all_records.append({
        "run_id": run.id,
        "model_type": config.get("model_type"),
        "datasets": config.get("datasets"),
        "n_features": config.get("n_features"),
        "odds_ratio": float(config.get("odds_ratio")),
        "control_frequency": float(config.get("control_frequency")),
        "save_dir": config.get("save_dir").replace("../../results/", "/mnt/disks/gmiller_data1/pnet/results/"),

        "perturbation_suffix": config.get("perturbation_suffix"),
        "perturbed_data_dir": config.get("perturbed_data_dir"),
        "target_f": os.path.join(config.get("perturbed_data_dir"), f"y_{config.get('perturbation_suffix')}.csv"),
    })

    for split in ["train", "validation", "test"]:
        all_records[-1][f"{split}_roc_auc_score"] = summary.get(f"{split}_roc_auc_score")
        all_records[-1][f"{split}_balanced_acc"] = summary.get(f"{split}_balanced_acc")
        all_records[-1][f"{split}_avg_precision"] = summary.get(f"{split}_average_precision_score")
        all_records[-1][f"{split}_acc"] = summary.get(f"{split}_acc")
        all_records[-1][f"{split}_f1_score"] = summary.get(f"{split}_f1_score")
        # all_records[-1][f"{split}_confusion_matrix"] = summary.get(f"{split}_confusion_matrix")
    for split in ["validation", "test"]:
        all_records[-1][f"{split}_feature_importances_path"] = os.path.join(all_records[-1]["save_dir"], f"{split}_gene_feature_importances.csv")
        all_records[-1][f"{split}_gene_importances_path"] = os.path.join(all_records[-1]["save_dir"], f"{split}_gene_importances.csv")

logging.debug("Convert to a single DataFrame")
df = pd.DataFrame(all_records)

logging.debug("Changing the column order for the final DataFrame")
group_by_cols = ["model_type", "n_features", "odds_ratio", "control_frequency"]
col_order = group_by_cols + performance_metric_col_order
df = df[col_order]

logging.debug("Make results DF collapsed over replicates")
df_group_summary = make_grouped_summary_with_mean_and_stdev(df, param_cols=group_by_cols, metric_cols=performance_metric_col_order)


logging.info(f"Saving the single-gene simulation results DataFrame to CSV at {single_gene_simu_result_savepath}")
df.to_csv(single_gene_simu_result_savepath, float_format="%.3f", index=False)
df_group_summary.to_csv(grouped_single_gene_simu_result_savepath, index=False)

display(df.round(3).head(2))
display(df_group_summary.head(10))

2025-09-03 19:00:52 INFO     [root] Working on single-gene simulation results: 360 runs found
2025-09-03 19:00:55 INFO     [root] Saving the single-gene simulation results DataFrame to CSV at ../results/single_gene_spike_in_simulation_prediction_metrics_per_run.csv


,model_type,n_features,odds_ratio,control_frequency,train_avg_precision,validation_avg_precision,test_avg_precision,train_roc_auc_score,validation_roc_auc_score,test_roc_auc_score,train_f1_score,validation_f1_score,test_f1_score,train_balanced_acc,validation_balanced_acc,test_balanced_acc,train_acc,validation_acc,test_acc
0,rf,100,30.0,0.5,0.754,0.664,0.712,0.790,0.714,0.783,0.741,0.68,0.723,0.718,0.628,0.726,0.718,0.629,0.726
1,rf,100,30.0,0.5,0.764,0.705,0.726,0.792,0.713,0.781,0.731,0.68,0.682,0.722,0.628,0.715,0.722,0.629,0.716


,model_type,n_features,odds_ratio,control_frequency,count,train_avg_precision,validation_avg_precision,test_avg_precision,train_roc_auc_score,validation_roc_auc_score,test_roc_auc_score,train_f1_score,validation_f1_score,test_f1_score,train_balanced_acc,validation_balanced_acc,test_balanced_acc,train_acc,validation_acc,test_acc
0,pnet,10,1.0,0.001,3,0.521 ± 0.0,0.48 ± 0.001,0.492 ± 0.014,0.511 ± 0.0,0.468 ± 0.001,0.486 ± 0.014,0.656 ± 0.014,0.621 ± 0.007,0.619 ± 0.005,0.525 ± 0.007,0.493 ± 0.006,0.509 ± 0.012,0.528 ± 0.007,0.479 ± 0.006,0.498 ± 0.012
1,pnet,10,1.0,0.010,3,0.523 ± 0.003,0.48 ± 0.0,0.473 ± 0.005,0.514 ± 0.002,0.468 ± 0.001,0.472 ± 0.006,0.622 ± 0.048,0.559 ± 0.088,0.569 ± 0.057,0.521 ± 0.004,0.466 ± 0.034,0.491 ± 0.005,0.523 ± 0.005,0.457 ± 0.028,0.484 ± 0.0
2,pnet,10,1.0,0.050,3,0.525 ± 0.003,0.487 ± 0.002,0.474 ± 0.006,0.513 ± 0.003,0.479 ± 0.007,0.471 ± 0.002,0.661 ± 0.017,0.631 ± 0.025,0.604 ± 0.002,0.521 ± 0.004,0.508 ± 0.003,0.481 ± 0.011,0.524 ± 0.004,0.494 ± 0.0,0.47 ± 0.012
3,pnet,10,1.0,0.100,3,0.522 ± 0.004,0.482 ± 0.037,0.469 ± 0.04,0.516 ± 0.006,0.463 ± 0.039,0.465 ± 0.047,0.284 ± 0.332,0.303 ± 0.288,0.205 ± 0.355,0.517 ± 0.011,0.504 ± 0.019,0.488 ± 0.008,0.516 ± 0.014,0.509 ± 0.028,0.495 ± 0.011
4,pnet,10,1.0,0.200,3,0.526 ± 0.004,0.463 ± 0.013,0.493 ± 0.006,0.518 ± 0.006,0.443 ± 0.02,0.479 ± 0.008,0.647 ± 0.024,0.615 ± 0.025,0.59 ± 0.032,0.523 ± 0.004,0.499 ± 0.003,0.487 ± 0.008,0.525 ± 0.005,0.487 ± 0.006,0.477 ± 0.006
5,pnet,10,1.0,0.500,3,0.515 ± 0.018,0.497 ± 0.08,0.491 ± 0.066,0.504 ± 0.02,0.462 ± 0.083,0.481 ± 0.071,0.327 ± 0.335,0.309 ± 0.327,0.296 ± 0.33,0.501 ± 0.002,0.488 ± 0.021,0.481 ± 0.032,0.501 ± 0.003,0.491 ± 0.023,0.484 ± 0.032
6,pnet,10,1.1,0.001,3,0.522 ± 0.001,0.481 ± 0.0,0.469 ± 0.003,0.512 ± 0.001,0.469 ± 0.001,0.467 ± 0.006,0.513 ± 0.268,0.484 ± 0.256,0.435 ± 0.31,0.518 ± 0.019,0.499 ± 0.004,0.491 ± 0.008,0.519 ± 0.022,0.494 ± 0.019,0.488 ± 0.006
7,pnet,10,1.1,0.010,3,0.522 ± 0.001,0.477 ± 0.007,0.485 ± 0.009,0.511 ± 0.002,0.459 ± 0.007,0.477 ± 0.008,0.665 ± 0.008,0.636 ± 0.016,0.615 ± 0.008,0.52 ± 0.004,0.501 ± 0.014,0.492 ± 0.021,0.523 ± 0.003,0.487 ± 0.013,0.481 ± 0.022
8,pnet,10,1.1,0.050,3,0.521 ± 0.002,0.486 ± 0.009,0.472 ± 0.014,0.513 ± 0.004,0.474 ± 0.011,0.476 ± 0.015,0.466 ± 0.336,0.45 ± 0.318,0.409 ± 0.354,0.522 ± 0.011,0.502 ± 0.01,0.497 ± 0.008,0.523 ± 0.014,0.498 ± 0.006,0.495 ± 0.011
9,pnet,10,1.1,0.100,3,0.523 ± 0.001,0.467 ± 0.005,0.47 ± 0.014,0.516 ± 0.002,0.436 ± 0.017,0.466 ± 0.014,0.522 ± 0.198,0.484 ± 0.165,0.45 ± 0.193,0.505 ± 0.022,0.479 ± 0.032,0.482 ± 0.02,0.506 ± 0.024,0.476 ± 0.034,0.481 ± 0.032


# Empirical results: P1000 somatic +/- germline

In [51]:
all_records = []
runs = api.runs(f"{PROJECT_NAME}", filters={"tags": "p1000-20250822"})
    # "$and": [{"tags": "pnet-simu-paper-20250822"}, {"tags": "1D-simu-20250822"}, {"tags": "2D-simu-20250822"}, {"tags": "p1000-20250822"}]})
logging.info(f"Working on empirical P1000 results: {len(runs)} runs found")

for run in runs: 
    config = run.config
    summary = run.summary
    all_records.append({
        "run_id": run.id,
        "model_type": config.get("model_type"),
        "datasets": config.get("datasets"),
        "save_dir": config.get("save_dir").replace("../../results/", "/mnt/disks/gmiller_data1/pnet/results/"),
        "input_data_dir": config.get("input_data_dir"),
    })

    for split in ["train", "validation", "test"]:
        all_records[-1][f"{split}_roc_auc_score"] = summary.get(f"{split}_roc_auc_score")
        all_records[-1][f"{split}_balanced_acc"] = summary.get(f"{split}_balanced_acc")
        all_records[-1][f"{split}_avg_precision"] = summary.get(f"{split}_average_precision_score")
        all_records[-1][f"{split}_acc"] = summary.get(f"{split}_acc")
        all_records[-1][f"{split}_f1_score"] = summary.get(f"{split}_f1_score")
        # all_records[-1][f"{split}_confusion_matrix"] = summary.get(f"{split}_confusion_matrix")
    for split in ["validation", "test"]:
        all_records[-1][f"{split}_feature_importances_path"] = os.path.join(all_records[-1]["save_dir"], f"{split}_gene_feature_importances.csv")
        all_records[-1][f"{split}_gene_importances_path"] = os.path.join(all_records[-1]["save_dir"], f"{split}_gene_importances.csv")

logging.debug("Convert to a single DataFrame")
df = pd.DataFrame(all_records)

logging.debug("Changing the column order for the final DataFrame")
group_by_cols = ["model_type","datasets"]
col_order = group_by_cols + performance_metric_col_order
# full_cols = col_order + [c for c in df.columns if c not in col_order]

df = df[col_order]
logging.debug("Make results DF collapsed over replicates")
df_group_summary = make_grouped_summary_with_mean_and_stdev(df, param_cols=group_by_cols, metric_cols=performance_metric_col_order)

logging.info(f"Saving the empirical P1000 results DataFrame to CSV at {p1000_result_savepath}")
df.to_csv(p1000_result_savepath, float_format="%.3f", index=False, )
df_group_summary.to_csv(grouped_p1000_result_savepath, index=False)

display(df.round(3).head(2))
display(df_group_summary.head(2))

2025-09-03 19:00:55 INFO     [root] Working on empirical P1000 results: 110 runs found
2025-09-03 19:00:56 INFO     [root] Saving the empirical P1000 results DataFrame to CSV at ../results/p1000_empirical_prediction_metrics_per_run.csv


,model_type,datasets,train_avg_precision,validation_avg_precision,test_avg_precision,train_roc_auc_score,validation_roc_auc_score,test_roc_auc_score,train_f1_score,validation_f1_score,test_f1_score,train_balanced_acc,validation_balanced_acc,test_balanced_acc,train_acc,validation_acc,test_acc
0,pnet,somatic_amp somatic_del somatic_mut germline_r...,0.995,0.849,0.918,0.996,0.931,0.940,0.973,0.821,0.717,0.977,0.870,0.781,0.983,0.888,0.842
1,pnet,somatic_amp somatic_del somatic_mut germline_r...,0.984,0.874,0.899,0.990,0.940,0.946,0.897,0.735,0.667,0.907,0.797,0.750,0.941,0.854,0.821


,model_type,datasets,count,train_avg_precision,validation_avg_precision,test_avg_precision,train_roc_auc_score,validation_roc_auc_score,test_roc_auc_score,train_f1_score,validation_f1_score,test_f1_score,train_balanced_acc,validation_balanced_acc,test_balanced_acc,train_acc,validation_acc,test_acc
0,pnet,germline_common_lof_missense,5,0.39 ± 0.001,0.341 ± 0.009,0.423 ± 0.003,0.584 ± 0.001,0.456 ± 0.002,0.586 ± 0.002,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,0.5 ± 0.0,0.5 ± 0.0,0.5 ± 0.0,0.684 ± 0.0,0.685 ± 0.0,0.663 ± 0.0
1,pnet,germline_rare_common_lof_missense,5,0.508 ± 0.022,0.532 ± 0.03,0.438 ± 0.049,0.661 ± 0.014,0.576 ± 0.01,0.576 ± 0.026,0.028 ± 0.051,0.028 ± 0.038,0.012 ± 0.027,0.507 ± 0.013,0.507 ± 0.01,0.503 ± 0.007,0.688 ± 0.008,0.69 ± 0.006,0.665 ± 0.005
